In [ ]:
###############################
## Não precisa rodar de novo ##
###############################

# !pip cache purge
# !pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 --no-cache-dir
# !pip install pillow, pandas

# !pip install -U transformers
# !pip install bitsandbytes
# !pip install accelerate

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration
from transformers import BitsAndBytesConfig
import torch


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

model_id = "/mnt/DADOS_GRENOBLE_1/laranjeira/VLMAge/models/LLaVA-1.5-7B-hf"
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, 
    dtype=torch.float16, 
    low_cpu_mem_usage=True, 
    quantization_config=quantization_config,
    device_map="cuda:0", 
)

processor = AutoProcessor.from_pretrained(model_id, use_fast=True)

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 3/3 [00:17<00:00,  5.75s/it]


In [3]:
import pandas as pd
df = pd.read_csv('../../LAGENDA/lagenda_annotation.csv')

imgs_todas_idades_ok = (
    df.groupby('img_name')['age']
      .apply(lambda x: (x != -1).all())
      .reset_index()
)

# Filtra as imagens em que TODAS as idades são conhecidas
imgs_todas_idades_ok = imgs_todas_idades_ok[imgs_todas_idades_ok['age'] == True]

# tem uma imagem faltando nos arquivos (olhar depois)
imgs_todas_idades_ok = imgs_todas_idades_ok[imgs_todas_idades_ok['img_name'] != 'lag_benchmark/b5c8ef090f134ad5.jpg.jpg']

print(f"{len(imgs_todas_idades_ok)} imagens têm todas as idades conhecidas.")
display(imgs_todas_idades_ok)

20182 imagens têm todas as idades conhecidas.


,img_name,age
0,lag_benchmark/0000339d0372e7e6.jpg,True
6,lag_benchmark/0004350a376865f5.jpg,True
9,lag_benchmark/00059fa95ee95e65.jpg,True
11,lag_benchmark/0007473cdbe8c18d.jpg,True
12,lag_benchmark/000851174d48d3d9.jpg,True
...,...,...
67154,lag_benchmark/fff74322fa07c7c1.jpg,True
67155,lag_benchmark/fff934bd1ff95af5.jpg,True
67156,lag_benchmark/fffb40d02f510b35.jpg,True
67157,lag_benchmark/ffff1ad4bf685255.jpg,True


In [4]:
prompts_mais_jovem = [
    """<image> Há alguma criança (menor de 14 anos) presente na imagem? Se sim, responda com 1. Caso contrário, responda com 0.
    ASSISTANT:""",
    """<image> Há alguma criança (menor de 18 anos) presente na imagem? Se sim, responda com 1. Caso contrário, responda com 0.
    ASSISTANT:""",
    """<image> Não sei.
    ASSISTANT:""",
    """<image> Não sei.
    ASSISTANT:"""
]

p1 = """USER: <image> 
Você recebe uma imagem que pode conter uma ou mais pessoas. Sua tarefa é determinar se cada pessoa na imagem tem menos de 13 anos.
        
Instruções:
- Classifique cada pessoa individualmente.
- Use APENAS os seguintes valores: ["0", "1"]
"1": Sim, a pessoa tem menos de 14 anos
"0": Não, a pessoa não tem menos de 14 anos

Requisitos de saída:
- Você DEVE responder APENAS com um JSON válido.
- NÃO escreva explicações, comentários ou texto fora do JSON.
Formato obrigatório:

{
  "numero de pessoas": X,
  "pessoas": [
      {
          "menor13": "um dos valores permitidos"
      }

  ]
}
ASSISTANT:"""

p2 = ""
p3 = ""
p4 = ""

prompts_todos = [
    p1,
    p2,
    p3,
    p4
]

## 10 imagens de cada vez

In [ ]:
prompt = """USER: <image>
Classifique cada pessoa na imagem com a seguinte definição:
- "menor18": idade menor que 18 anos
- "maior18": idade maior ou igual a 18 anos

Regras:
- Responda APENAS com JSON válido
- Nenhum texto extra antes ou depois

Formato de saída:
{
  "número de pessoas": <inteiro>,
  "pessoas": [
    {"id": 1, "categoria": <classificação da idade> }, 
  ]
}
ASSISTANT:"""

In [5]:
import os
import json
import requests
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm  # <-- barra de progresso
from codecarbon import EmissionsTracker

########## code carbon ##########
tracker = EmissionsTracker(tracking_mode='process', log_level='critical', output_file='emissions_llava1.5.csv')
tracker.start()
#################################

# pasta_imagens = "../../LAGENDA/images"
pasta_imagens = "../../LAGENDA/images_noface"

prompt = prompts_mais_jovem[1]
resultados = []
batch_size = 10
cont = 0

torch.cuda.empty_cache()
for i in tqdm(range(0, len(imgs_todas_idades_ok), batch_size), desc="Processando imagens"):
    batch_imgs = imgs_todas_idades_ok['img_name'][i:i+batch_size]
    
    batch_images = []
    batch_nomes_arquivo = []
    
    for img_name in batch_imgs:
        nome_arquivo = img_name.split("/")[-1]
        batch_nomes_arquivo.append(nome_arquivo)
        caminho = f"{pasta_imagens}/{nome_arquivo}"
        imagem = Image.open(caminho)
        batch_images.append(imagem)
    
    inputs = processor(
        text=[prompt] * len(batch_images),  # Same prompt for all
        images=batch_images,
        return_tensors="pt",
        padding=True  # Important for batching
    ).to(0)
    
    # Generate for entire batch
    outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    generated_texts = processor.batch_decode(outputs, skip_special_tokens=True)
    
    # Process results
    for img_name, generated_text in zip(batch_nomes_arquivo, generated_texts):
        resposta = generated_text.split("ASSISTANT:")[-1].strip().lower()
        # dados = json.loads(resposta_json)
        # resposta_final = [p["menor13"] for p in dados["pessoas"]]
        resultados.append({
            "img_name": img_name,
            "resposta_modelo": resposta
        })
    

    del inputs, outputs
    torch.cuda.empty_cache()

########## code carbon ##########
emissions = tracker.stop()
print(f"Emissions: {emissions} kg CO₂")
#################################
    
# visualizar
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

df_resultados.to_csv('resultados_llava1.5.csv', index=False)

idades_por_imagem = (
    df.groupby('img_name')['age']
      .apply(list)
      .reset_index()  
)

# Junta com sua tabela atual (que já tem só as imagens válidas)
# imgs_com_idades = imgs_todas_idades_ok.merge(idades_por_imagem, on='img_name')

# Mostra
#display(imgs_com_idades)

[codecarbon WARNING @ 23:47:34] Multiple instances of codecarbon are allowed to run at the same time.
Processando imagens: 100%|████████████████████████████████████████████████████████| 2019/2019 [6:08:31<00:00, 10.95s/it]

Emissions: 0.1416830124205491 kg CO₂


,img_name,resposta_modelo
0,0000339d0372e7e6.jpg,0
1,0004350a376865f5.jpg,1
2,00059fa95ee95e65.jpg,0
3,0007473cdbe8c18d.jpg,1
4,000851174d48d3d9.jpg,1
...,...,...
20177,fff74322fa07c7c1.jpg,0
20178,fff934bd1ff95af5.jpg,0
20179,fffb40d02f510b35.jpg,1
20180,ffff1ad4bf685255.jpg,0


NameError: name 's' is not defined

In [6]:
resultados

[{'img_name': '0000339d0372e7e6.jpg', 'resposta_modelo': '0'},
 {'img_name': '0004350a376865f5.jpg', 'resposta_modelo': '1'},
 {'img_name': '00059fa95ee95e65.jpg', 'resposta_modelo': '0'},
 {'img_name': '0007473cdbe8c18d.jpg', 'resposta_modelo': '1'},
 {'img_name': '000851174d48d3d9.jpg', 'resposta_modelo': '1'},
 {'img_name': '000de3ca54bd4c1a.jpg', 'resposta_modelo': '0'},
 {'img_name': '00107eaf96d165ca.jpg', 'resposta_modelo': '0'},
 {'img_name': '001542bf7f36af27.jpg', 'resposta_modelo': '0'},
 {'img_name': '0019ce21ce6df11a.jpg', 'resposta_modelo': '1'},
 {'img_name': '001a0baa79925adb.jpg', 'resposta_modelo': '0'},
 {'img_name': '001a512c540796dc.jpg', 'resposta_modelo': '0'},
 {'img_name': '001bf60cd96147ea.jpg', 'resposta_modelo': '1'},
 {'img_name': '002151c4bde6ca15.jpg', 'resposta_modelo': '0'},
 {'img_name': '0024a6207b572a5b.jpg', 'resposta_modelo': '1'},
 {'img_name': '00289f80c48c170e.jpg', 'resposta_modelo': '0'},
 {'img_name': '0028cac8669a9712.jpg', 'resposta_modelo'

## 1 imagem de cada vez

In [4]:
import os
import json
import requests
from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tqdm import tqdm  # <-- barra de progresso
from codecarbon import EmissionsTracker

########## code carbon ##########
tracker = EmissionsTracker(tracking_mode='process', log_level='critical', output_file='emissions_llava1.5.csv')
tracker.start()
#################################


resultados = []
cont = 0

for img_name in tqdm(imgs_todas_idades_ok['img_name'], desc="Processando imagens"):
    nome_arquivo = img_name.split("/")[-1]

    # monta o caminho da imagem automaticamente
    caminho = f"../LAGENDA/images/{nome_arquivo}"

    # abre a imagem
    imagem = Image.open(caminho)

    # mesmo prompt para todas
    prompt = """<image> Há alguma criança (menor de 14 anos) presente na imagem? Se sim, responda com 1. Caso contrário, responda com 0.
    ASSISTANT:"""

    # prepara entrada
    inputs = processor(
        text=prompt,
        images= imagem,
        return_tensors="pt"
    ).to(0)

    # gera resposta
    output = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    generated_text = processor.batch_decode(output, skip_special_tokens=True)
    resposta = generated_text[0].split("ASSISTANT:")[-1].strip().lower()

    # dados = json.loads(resposta)
    # respf = [p["age"] for p in dados["people"]]
    
    # # salva resultado
    resultados.append({
        "img_name": img_name,
        "resposta_modelo": resposta
    })
    cont += 1

    del inputs, outputs
    torch.cuda.empty_cache()

########## code carbon ##########
emissions = tracker.stop()
print(f"Emissions: {emissions} kg CO₂")
#################################
    
# visualizar
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

df_resultados.to_csv('resultados_llava1.5.csv', index=False)

idades_por_imagem = (
    df.groupby('img_name')['age']
      .apply(list)
      .reset_index()  
)

# Junta com sua tabela atual (que já tem só as imagens válidas)
# imgs_com_idades = imgs_todas_idades_ok.merge(idades_por_imagem, on='img_name')

# Mostra
#display(imgs_com_idades)

[codecarbon WARNING @ 19:24:09] Multiple instances of codecarbon are allowed to run at the same time.
Processando imagens:   2%|▊                                   | 424/20183 [10:24<8:05:16,  1.47s/it]


KeyboardInterrupt: 

In [9]:


creds, _ = default()
gc = gspread.authorize(creds)

planilha = gc.open_by_url("https://docs.google.com/spreadsheets/d/1UR7QOIrhS421fftov0DsU_Rzz__CjmBhfFkUujCoHyk/edit?usp=sharing")
aba = planilha.sheet1
coluna = df_resultados['resposta_modelo'].apply(lambda x: ','.join(x)).tolist()

aba.update(
    values=[[v] for v in coluna],
    range_name=f'B4:B{len(coluna)+3}'
)

ModuleNotFoundError: No module named 'gspread'

In [8]:
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

,img_name,resposta_modelo
0,lag_benchmark/0000339d0372e7e6.jpg,0
1,lag_benchmark/0004350a376865f5.jpg,1
2,lag_benchmark/00059fa95ee95e65.jpg,0
3,lag_benchmark/0007473cdbe8c18d.jpg,1
4,lag_benchmark/000851174d48d3d9.jpg,1
...,...,...
6315,lag_benchmark/502bff235463e8ac.jpg,1
6316,lag_benchmark/502e1bf1debb5f0d.jpg,0
6317,lag_benchmark/5031ea12a77b5299.jpg,0
6318,lag_benchmark/50357b2bc521846d.jpg,0


In [3]:
import requests
from PIL import Image

image2 = Image.open(requests.get("https://media.istockphoto.com/id/2087707686/pt/foto/group-of-asian-woman-friends-playing-together-at-the-beach.jpg?s=612x612&w=0&k=20&c=snVydWBhGDK7SV-oYJmzkz-8nuvDukrPu7tSAAJcRUc=", stream=True).raw)

In [4]:
prompt = '''USER: <image>
You are given an image that may contain one or more people.

Your task is to estimate the age range of EACH visible person.

Instructions:
- Assign one age category per person.
- Use ONLY the following categories:
  ["child", "teen", "adult", "older"]
- "child": ~0–13
- "teen": ~14–19
- "adult": ~20-59
- "older": 60+

Output requirements:
- Return ONLY valid JSON.
- Do NOT include explanations or extra text.

JSON format:
{
  "number of people": X,
  "people": [
    {
      "id": 1,
      "agecategory": "one of the allowed labels"
    }
  ]
}
ASSISTANT:
'''
inputs = processor(text=prompt, images=image2, return_tensors="pt").to("cuda")
for k,v in inputs.items():
  print(k,v.shape)

input_ids torch.Size([1, 779])
attention_mask torch.Size([1, 779])
pixel_values torch.Size([1, 3, 336, 336])


In [5]:
output = model.generate(**inputs, max_new_tokens=500)
generated_text = processor.batch_decode(output, skip_special_tokens=True)
for text in generated_text:
  print(text.split("ASSISTANT:")[-1])


{
"number of people": 3,
"people": [
{
"id": 1,
"agecategory": "adult"
},
{
"id": 2,
"agecategory": "adult"
},
{
"id": 3,
"agecategory": "adult"
}
]
}


In [11]:
import os
import pandas as pd

pasta_dataset = 'LAGENDA'
df = pd.read_csv(os.path.join(pasta_dataset,'lagenda_annotation.csv'))

for image in os.listdir(os.path.join(pasta_dataset, 'images' )):
    print(image)
    break

48094373ae94fec1.jpg
